In [ ]:
import json

# Load intents from JSON file
with open('intents.json', 'r') as file:
    intents = json.load(file)

print(intents)

{'intents': [{'tag': 'greeting', 'patterns': ['Hi', 'Selamat', 'Is anyone there?', 'Hello', 'Assalammualaikum', 'Aslkm', 'Salam'], 'responses': ['Hi, welcome to Nadnie Homestay!', 'Hi, how can I assist you?', 'Hello, welcome to Nadnie Homestay. How can I assist you?', 'Waalaikumussalam, how can I help you today?', 'Yes, how can I help you?']}, {'tag': 'goodbye', 'patterns': ['Bye', 'See you next time', 'Goodbye'], 'responses': ['Thank you for visiting Nadnie Homestay, see you next time!', 'Have a nice day', 'Bye! Come again soon!']}, {'tag': 'thanks', 'patterns': ['Thanks', 'Thank you', 'Thank you for your service'], 'responses': ['Happy to help!', 'Any time!', 'My pleasure~', 'You are most welcome!']}, {'tag': 'check in', 'patterns': ['check in', 'What time can I check in?', 'When can I check in?', 'When does check in start?'], 'responses': ['Check in starts at 2 pm', 'You can check in your room by 2 pm', 'Please check in by 2 pm', 'Sir/Madam, you can check in by 2 pm']}, {'tag': 'roo

**DATA PREPROCESSING**

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string
import json

# Download necessary NLTK resources
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')

# Initialize the stemmer
stemmer = PorterStemmer()

def preprocess_input(user_input):
    # Convert to lowercase
    user_input = user_input.lower()

    # Replace "check in" and "check out" with unique tokens
    user_input = user_input.replace("check in", "check_in").replace("check out", "check_out")

    # Remove punctuation
    user_input = user_input.translate(str.maketrans('', '', string.punctuation))

    # Tokenize the input
    tokens = word_tokenize(user_input)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    stop_words.update(string.punctuation)
    filtered_tokens = [word for word in tokens if word not in stop_words]

    # Apply stemming
    stemmed_tokens = [stemmer.stem(word) for word in filtered_tokens]

    return stemmed_tokens

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


**RULE BASED**

In [ ]:
import random

# Load responses from JSON file
def load_responses(filename):
    with open('intents.json', 'r') as file:
        return json.load(file)

# Chatbot response function
def chatbot_response(user_input, responses):
    processed_input = preprocess_input(user_input)

    # Handling special greetings like giving salam
    special_greetings = ["assalammualaikum", "aslkm", "salam"]
    if any(greeting in processed_input for greeting in special_greetings):
        return "Waalaikumussalam, how can I help you today?"

    # Check for keywords in the responses
    for intent in responses['intents']:
        for pattern in intent['patterns']:
            processed_pattern = preprocess_input(pattern)

            # Ensure a more specific match (use exact match or improved partial match logic)
            if processed_input == processed_pattern or processed_input in processed_pattern:
                return random.choice(intent['responses'])

    return responses.get("default_response", "I'm sorry, I didn't understand that.")  # Default response if no match

# Load responses from the JSON file
responses = load_responses('intents.json')

# Command-line interface
while True:
    user_input = input("You: ")
    if user_input.lower() == "end chat":
        print("Chatbot: Thank you for reaching out! Have a great day!")
        break
    response = chatbot_response(user_input, responses)
    print("Chatbot:", response)

You: Assalammualaikum
Chatbot: Waalaikumussalam, how can I help you today?
You: hello
Chatbot: Hi, how can I assist you?
You: How many rooms are in this homestay?
Chatbot: Our homestay has 3 bedrooms and 3 bathrooms.
You: What are the facilities provided?
Chatbot: We provide a private pool, BBQ area and karaoke room.
You: How many people can this homestay accommodate
Chatbot: The maximum number of pax is 10 people for the sake of your comfort.
You: How can i book a room?
Chatbot: For same day booking, please call 0115966123.
You: do you accept QR
Chatbot: We accept most renowned credit cards
You: what time can i check in?
Chatbot: You can check in your room by 2 pm
You: By what time do i have to check out?
Chatbot: You can check out by 1 pm.
You: Thank you for your service
Chatbot: Happy to help!
Chatbot: My pleasure~
You: Goodbye
Chatbot: Thank you for visiting Nadnie Homestay, see you next time!
You: end chat
Chatbot: Thank you for reaching out! Have a great day!


**ML BASED**

In [ ]:
import json
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('wordnet')

# Load intents from JSON file
def load_intents(filename):
    with open(filename, 'r') as file:
        return json.load(file)

# Preprocess the data
def preprocess_data(intents):
    lemmatizer = WordNetLemmatizer()
    training_sentences = []
    training_labels = []
    responses = {}

    for intent in intents['intents']:
        for pattern in intent['patterns']:
            # Tokenize and lemmatize each word
            word_list = nltk.word_tokenize(pattern)
            word_list = [lemmatizer.lemmatize(word.lower()) for word in word_list]
            training_sentences.append(' '.join(word_list))
            training_labels.append(intent['tag'])

        # Store responses for later use
        responses[intent['tag']] = intent['responses']

    return training_sentences, training_labels, responses

# Load intents
intents = load_intents('intents.json')

# Preprocess data
X, y, responses = preprocess_data(intents)

# Print the lengths of the original data
print(f"Original data length: {len(X)}")
print(f"Original labels length: {len(y)}")

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Print the lengths after encoding
print(f"Encoded labels length: {len(y_encoded)}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Print the lengths of the training data
print(f"Length of X_train: {len(X_train)}")
print(f"Length of y_train: {len(y_train)}")

# Create a simple text vectorization layer
vectorizer = layers.TextVectorization(output_mode='int', max_tokens=1000)

# Adapt the vectorizer to the training data
vectorizer.adapt(X_train)

# Vectorize the training and testing data
X_train_vectorized = vectorizer(np.array(X_train)).numpy()
X_test_vectorized = vectorizer(np.array(X_test)).numpy()

# Build the model
model = keras.Sequential([
    layers.Input(shape=(None,), dtype=tf.int64),
    layers.Embedding(input_dim=1000, output_dim=64),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(label_encoder.classes_), activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train_vectorized, y_train, epochs=10, validation_data=(X_test_vectorized, y_test))

# Function to get a response based on user input
def get_response(user_input):
    # Tokenize and lemmatize the input
    user_input = nltk.word_tokenize(user_input)
    user_input = ' '.join([WordNetLemmatizer().lemmatize(word.lower()) for word in user_input])
    # Vectorize the input
    user_input_vectorized = vectorizer([user_input]).numpy()
    # Predict intent
    prediction = model.predict(user_input_vectorized)
    intent_index = np.argmax(prediction)
    intent_tag = label_encoder.inverse_transform([intent_index])[0]
    return np.random.choice(responses[intent_tag])

# Command-line interface
print("Chatbot is ready! Type 'exit' to end the chat.")
while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break
    response = get_response(user_input)
    print("Chatbot:", response)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...


Original data length: 43
Original labels length: 43
Encoded labels length: 43
Length of X_train: 34
Length of y_train: 34
Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 205ms/step - accuracy: 0.0901 - loss: 2.3066 - val_accuracy: 0.0000e+00 - val_loss: 2.3004
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.0901 - loss: 2.2992 - val_accuracy: 0.1111 - val_loss: 2.2957
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.1501 - loss: 2.2962 - val_accuracy: 0.2222 - val_loss: 2.2927
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.1998 - loss: 2.2934 - val_accuracy: 0.4444 - val_loss: 2.2914
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3002 - loss: 2.2912 - val_accuracy: 0.3333 - val_loss: 2.2905
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.3799 - loss: 2.2893 - val_accuracy: 0.3333 - val_loss: 2.2885
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3799 - loss: 2.2866 - val_accuracy: 0.5556 - val_loss: 2.285